# THEMIS ESA Plasma Instrument — Implementation / 구현

**Paper**: McFadden, J. P., Carlson, C. W., Larson, D., Ludlam, M., Abiad, R., Elliott, B., Turin, P., Marckwordt, M., & Angelopoulos, V. (2008). *The THEMIS ESA Plasma Instrument and In-flight Calibration*. Space Science Reviews, 141, 277–302. DOI: 10.1007/s11214-008-9440-2

This notebook reproduces the **four core building blocks** of the THEMIS ESA data pipeline using synthetic plasma data on top of NumPy / SciPy / Matplotlib. / 이 노트북은 THEMIS ESA 자료 처리 파이프라인의 **네 가지 핵심 구성요소**를 합성 플라즈마 데이터와 NumPy / SciPy / Matplotlib로 재현한다.

1. **Top-hat ESA velocity-space response** — logarithmic energy bins + 16 azimuthal sectors on a hemisphere / 로그 에너지 빈과 16개 방위 분할.
2. **In-flight cross-calibration** — match iESA / eESA densities in a synthetic magnetosheath crossing to recover relative efficiency / 자기초 통과 자료에서 이온–전자 밀도 일치로 상대 효율 복원.
3. **VDF moment computation** — discretized integrals for n, u, P with energy-bin weighting and dead-time / spacecraft-potential corrections / 에너지 빈 가중 + 데드타임 / 위성 전위 보정 포함 모멘트 계산.
4. **Plasma sheet vs lobe population separation** — temperature / density classifier on a synthetic boundary crossing / 온도 / 밀도 분류기로 자기꼬리 경계 분리.

The full ESA data pipeline involves 1024 measurements per spacecraft per spin across ten sensors; here we focus on the **methodology** with reduced-size synthetic distributions. / 실제 파이프라인은 위성당 회전마다 1024 측정이지만 여기서는 합성 분포로 **방법론**에 집중한다.

In [ ]:
# Imports and global plotting setup.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.optimize import minimize_scalar

np.random.seed(20260425)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Physical constants (SI for clarity, then convert as needed).
M_PROTON = 1.6726e-27   # kg
M_ELECTRON = 9.1094e-31 # kg
Q_E = 1.6022e-19        # C
EV_TO_J = Q_E           # 1 eV in Joules
KB = 1.3807e-23         # J/K

## Part 1: Top-Hat ESA Velocity-Space Response / 탑햇 ESA 속도 공간 응답

The THEMIS ESA selects energy via concentric hemispheres at the analyzer constant $k$ ($E/q = k V_{\rm inner}$, $k_{\rm iESA} = 6.2$, $k_{\rm eESA} = 7.9$). A logarithmic sweep covers $\sim$6 eV to 25–32 keV in 32 steps, while the 3-second spin paints a full $4\pi$ sr by combining 16 azimuthal sectors with the polar anode array (16 anodes for ions, 8 for electrons). / THEMIS ESA는 동심 반구 사이 전압으로 에너지를 선택하고($E/q = k V_{\rm inner}$, $k_{\rm iESA}=6.2$, $k_{\rm eESA}=7.9$), 32 스텝 로그 스윕(약 6 eV–25/32 keV)과 3초 회전(16 방위 × 8/16 양극)으로 $4\pi$ sr을 덮는다.

We construct the energy/angle grid, the geometric factor per pixel, and visualize the resulting response on a synthetic isotropic Maxwellian. / 에너지/각도 격자, 픽셀별 기하 인자, 등방 맥스웰 분포에 대한 응답을 구성·시각화한다.

In [ ]:
def make_esa_grid(species='ion', n_energies=32, n_azimuth=16):
    """Construct the THEMIS ESA energy/angle bin centers and edges.

    Args:
        species: 'ion' or 'electron'. Sets energy range and polar anode count.
        n_energies: Number of logarithmic energy steps per spin.
        n_azimuth: Number of azimuthal sectors over a 3 s spin.

    Returns:
        Dict with energy centers/edges (eV), polar anode centers/edges (rad),
        azimuth centers/edges (rad), and total geometric factor for one pixel.
    """
    if species == 'ion':
        e_min, e_max = 6.0, 25_000.0
        n_polar = 16
        g_total = 0.0061  # cm^2 sr eV (in-flight, McFadden Table 1)
    else:
        e_min, e_max = 6.0, 32_000.0
        n_polar = 8
        g_total = 0.0066

    # Logarithmic energy grid: 32 bin centers + edges.
    log_e = np.linspace(np.log(e_max), np.log(e_min), n_energies)
    energies = np.exp(log_e)
    log_edges = np.linspace(np.log(e_max) + 0.5 * (log_e[0] - log_e[1]),
                            np.log(e_min) - 0.5 * (log_e[0] - log_e[1]),
                            n_energies + 1)
    e_edges = np.exp(log_edges)

    # Polar anodes: equal-angle on 180 deg.
    polar_edges = np.linspace(0.0, np.pi, n_polar + 1)
    polar_centers = 0.5 * (polar_edges[:-1] + polar_edges[1:])

    # Azimuth sectors (full 2*pi over a spin).
    az_edges = np.linspace(0.0, 2 * np.pi, n_azimuth + 1)
    az_centers = 0.5 * (az_edges[:-1] + az_edges[1:])

    # Per-pixel geometric factor: total G shared across pixels.
    g_pixel = g_total / (n_polar * n_azimuth)

    return {
        'species': species,
        'energies': energies,
        'energy_edges': e_edges,
        'polar_centers': polar_centers,
        'polar_edges': polar_edges,
        'az_centers': az_centers,
        'az_edges': az_edges,
        'g_total': g_total,
        'g_pixel': g_pixel,
    }


grid_i = make_esa_grid('ion')
grid_e = make_esa_grid('electron')
print(f"iESA energy span: {grid_i['energies'][-1]:.2f} eV  ->  {grid_i['energies'][0]:.0f} eV")
print(f"eESA energy span: {grid_e['energies'][-1]:.2f} eV  ->  {grid_e['energies'][0]:.0f} eV")
print(f"iESA pixel G: {grid_i['g_pixel']:.4e} cm^2 sr eV (16 polar x 16 azimuth)")
print(f"eESA pixel G: {grid_e['g_pixel']:.4e} cm^2 sr eV (8 polar x 16 azimuth)")

In [ ]:
def maxwellian_f(v, n, T_eV, mass):
    """Isotropic Maxwellian phase-space density f(v) in s^3 / cm^6.

    Args:
        v: Speed in cm/s.
        n: Number density in cm^-3.
        T_eV: Temperature in eV.
        mass: Particle mass in kg.

    Returns:
        f(v) in s^3 / cm^6.
    """
    T_J = T_eV * EV_TO_J
    v_th2 = 2.0 * T_J / mass         # m^2/s^2
    v_th2_cgs = v_th2 * 1e4          # convert to cm^2/s^2
    norm = n * (mass / (2 * np.pi * T_J)) ** 1.5
    norm_cgs = norm * 1e-6           # per cm^3 -> s^3/cm^6
    return norm_cgs * np.exp(-(v ** 2) / v_th2_cgs)


def f_to_counts(f, energies_eV, g_pixel, mass, dt_acc=6e-3, eff_E=None):
    """Convert phase-space density f(E) to expected counts per energy bin.

    Uses C = G * j * dt with j = 2 E^2 f / m^2 (differential flux). McFadden
    eqs. 4.2 and 4.3 give counts per pixel per accumulation interval.

    Args:
        f: Phase-space density (s^3/cm^6) at each energy bin.
        energies_eV: Bin-center energies (eV).
        g_pixel: Geometric factor for one pixel (cm^2 sr eV).
        mass: Particle mass (kg).
        dt_acc: Accumulation time per energy step (s, default 6 ms).
        eff_E: Optional energy-dependent efficiency multiplier.

    Returns:
        Expected counts per pixel per energy step.
    """
    if eff_E is None:
        eff_E = np.ones_like(energies_eV)
    E_J = energies_eV * EV_TO_J
    # Differential flux in proportional units; absolute scale absorbed in G.
    j = (2.0 * E_J / (mass * 1e3) ** 2) * f * 1e-7
    counts = g_pixel * j * dt_acc * eff_E * energies_eV
    return np.maximum(counts, 0.0)


# Sample magnetosheath ions (n=20 cm^-3, T=200 eV).
v_grid = np.sqrt(2.0 * grid_i['energies'] * EV_TO_J / M_PROTON) * 1e2  # cm/s
f_i = maxwellian_f(v_grid, n=20.0, T_eV=200.0, mass=M_PROTON)
counts_i = f_to_counts(f_i, grid_i['energies'], grid_i['g_pixel'], M_PROTON)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].loglog(grid_i['energies'], f_i, 'o-')
axes[0].set_xlabel('Energy [eV]'); axes[0].set_ylabel('f(v) [s^3 / cm^6]')
axes[0].set_title('Synthetic magnetosheath ion VDF / 합성 자기초 이온 VDF')
axes[0].grid(True, which='both', alpha=0.3)

axes[1].loglog(grid_i['energies'], counts_i + 1e-6, 's-', color='C1')
axes[1].set_xlabel('Energy [eV]'); axes[1].set_ylabel('Counts / pixel / step')
axes[1].set_title('Expected counts per pixel / 픽셀당 기대 계수')
axes[1].grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

## Part 2: In-Flight Cross-Calibration via Magnetosheath Ion Peak / 자기초 이온 피크를 이용한 비행 중 교차 교정

McFadden et al. (§2.5) anchor the calibration on a magnetosheath crossing where the iESA-derived $N_i$ and the eESA-derived $N_e$ should agree (after dead-time and spacecraft-potential corrections, and after accounting for solar-wind alpha contamination). The procedure: / McFadden 등(§2.5)은 자기초 통과 시 데드타임·위성 전위·알파 보정 후 $N_i$와 $N_e$가 일치해야 한다는 관계를 기준으로 교정한다. 절차는

1. Generate synthetic ion and electron count spectra with a *known* relative efficiency mismatch. / 알려진 상대 효율 불일치를 가진 합성 이온·전자 계수 스펙트럼 생성.
2. Compute crude $N_i$, $N_e$ moments. / 임시 모멘트 계산.
3. Solve for $\varepsilon_{\rm rel}^{\rm i}$ that forces $N_i / N_e \to 1$. / $N_i / N_e \to 1$이 되도록 $\varepsilon_{\rm rel}^{\rm i}$ 추정.
4. Inject ~170 ns dead-time and demonstrate the unphysical $N_i / N_e > 1$ → ~0.9 recovery (Fig. 12 of the paper). / 170 ns 데드타임을 주입해 $N_i/N_e > 1$이 보정 후 ~0.9로 복원되는 Fig. 12 결과를 재현.

In [ ]:
def density_from_omni_counts(counts, energies_eV, g_pixel, mass, n_pixels, dt_acc=6e-3):
    """Estimate omnidirectional density n by summing f over energy bins.

    Using v^2 dv = (1/2) sqrt(2/m) E^{1/2} dE in SI, but operating in cm/s here:
    n = sum_E 4*pi v^2 dv f(E). For a logarithmic grid, dv/v = (1/2) dE/E.

    Args:
        counts: Counts per pixel per energy step, shape (n_E,) (omni-summed).
        energies_eV: Bin-center energies (eV).
        g_pixel: Per-pixel geometric factor (cm^2 sr eV).
        mass: Particle mass (kg).
        n_pixels: Total pixels integrated (so total G = n_pixels * g_pixel).
        dt_acc: Accumulation time per step (s).

    Returns:
        Density estimate (cm^-3).
    """
    E_J = energies_eV * EV_TO_J
    # Recover f from counts (inverse of f_to_counts).
    j = counts / (g_pixel * energies_eV * dt_acc * n_pixels)
    f = j / ((2.0 * E_J / (mass * 1e3) ** 2) * 1e-7)
    v_cgs = np.sqrt(2.0 * E_J / mass) * 1e2  # cm/s
    dlogE = np.abs(np.gradient(np.log(energies_eV)))
    dv = 0.5 * v_cgs * dlogE
    n = np.sum(4 * np.pi * v_cgs ** 2 * f * dv)
    return n


# Build synthetic magnetosheath: same n, T for ions and electrons (other than alpha).
N_TRUE = 20.0   # cm^-3
T_I = 200.0     # eV
T_E = 30.0      # eV (electrons cooler than ions in sheath)

# Ion grid and counts (with TRUE relative efficiency = 1.0).
v_i = np.sqrt(2.0 * grid_i['energies'] * EV_TO_J / M_PROTON) * 1e2
f_i = maxwellian_f(v_i, N_TRUE, T_I, M_PROTON)
counts_i_true = f_to_counts(f_i, grid_i['energies'], grid_i['g_total'], M_PROTON)

v_e = np.sqrt(2.0 * grid_e['energies'] * EV_TO_J / M_ELECTRON) * 1e2
f_e = maxwellian_f(v_e, N_TRUE, T_E, M_ELECTRON)
counts_e_true = f_to_counts(f_e, grid_e['energies'], grid_e['g_total'], M_ELECTRON)

# Inject mismatch: ion sensor secretly under-efficient by 12%.
EPS_MISMATCH = 0.88
counts_i_obs = counts_i_true * EPS_MISMATCH

n_i_naive = density_from_omni_counts(counts_i_obs, grid_i['energies'],
                                     grid_i['g_total'], M_PROTON, n_pixels=1)
n_e_naive = density_from_omni_counts(counts_e_true, grid_e['energies'],
                                     grid_e['g_total'], M_ELECTRON, n_pixels=1)
print(f"Pre-cal:  Ni = {n_i_naive:.2f} cm^-3,  Ne = {n_e_naive:.2f} cm^-3,  Ni/Ne = {n_i_naive / n_e_naive:.3f}")

# Solve for relative efficiency that forces Ni == Ne.
def density_residual(eps):
    """Squared log-ratio between ion and electron densities at trial eps."""
    n_i = density_from_omni_counts(counts_i_obs / eps, grid_i['energies'],
                                   grid_i['g_total'], M_PROTON, n_pixels=1)
    return (np.log(n_i) - np.log(n_e_naive)) ** 2


res = minimize_scalar(density_residual, bounds=(0.5, 1.5), method='bounded')
eps_recovered = res.x
n_i_post = density_from_omni_counts(counts_i_obs / eps_recovered, grid_i['energies'],
                                    grid_i['g_total'], M_PROTON, 1)
print(f"Recovered iESA relative efficiency: {eps_recovered:.4f} (true {EPS_MISMATCH:.4f})")
print(f"Post-cal Ni/Ne: {n_i_post / n_e_naive:.3f}")

### Dead-time correction in dense plasma / 고밀도에서의 데드타임 보정

McFadden Fig. 12 shows that in the magnetosheath the *uncorrected* $N_i / N_e$ can drift to 1.0–1.3, an unphysical value that the 170 ns electronic dead time of the Amptek A121 pre-amp explains entirely. We reproduce this with a high-rate synthetic spectrum. / Fig. 12은 자기초에서 보정 전 $N_i/N_e$가 1.0–1.3까지 올라가는 비물리적 값을, 170 ns 전자 데드타임으로 설명함을 보인다. 합성 고계수 스펙트럼으로 재현한다.

In [ ]:
def apply_deadtime(true_rate_hz, tau=170e-9):
    """Apply non-paralyzable dead-time loss: C_m = C_t / (1 + C_t tau)."""
    return true_rate_hz / (1.0 + true_rate_hz * tau)


def remove_deadtime(measured_rate_hz, tau=170e-9):
    """Invert dead-time: C_t = C_m / (1 - C_m tau)."""
    return measured_rate_hz / (1.0 - measured_rate_hz * tau)


# Build per-pixel rate spectrum scaled into the MHz regime where dead-time is
# the dominant systematic (McFadden Fig. 12 case).
DT_ACC = 6e-3  # 6 ms accumulation per energy step
true_rate_i = counts_i_true / DT_ACC * 5.0e6  # boost to several MHz at the peak
measured_rate_i = apply_deadtime(true_rate_i)
rate_corrected = remove_deadtime(measured_rate_i)

print(f"Peak true rate:                              {true_rate_i.max() / 1e6:.2f} MHz")
print(f"Peak measured rate (with 170 ns dead-time):  {measured_rate_i.max() / 1e6:.2f} MHz")
print(f"Peak rate after dead-time correction:        {rate_corrected.max() / 1e6:.2f} MHz")
ratio_uncorr = (measured_rate_i / true_rate_i)[true_rate_i > 1e3]
ratio_corr = (rate_corrected / true_rate_i)[true_rate_i > 1e3]
print(f"Mean (measured/true) - lost fraction:        {1 - ratio_uncorr.mean():.3f}")
print(f"Mean (corrected/true) - residual after fix:  {1 - ratio_corr.mean():.3e}")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.loglog(grid_i['energies'], true_rate_i / 1e6, 'k-', label='true / 참값')
ax.loglog(grid_i['energies'], measured_rate_i / 1e6, 'C3o-', label=r'measured ($\tau$=170 ns) / 측정')
ax.loglog(grid_i['energies'], rate_corrected / 1e6, 'C2s--', label='dead-time corrected / 보정')
ax.set_xlabel('Energy [eV]'); ax.set_ylabel('Count rate per pixel [MHz]')
ax.set_title('Dead-time effect at MHz rates / MHz 영역에서의 데드타임 효과')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

## Part 3: VDF Moment Computation with Weighting / 가중치를 포함한 VDF 모멘트 계산

McFadden §2 (and Eq. 4.4 of the notes) gives the discrete moments / McFadden §2 (그리고 노트 4.4)는 다음 이산 모멘트를 정의한다:

$$
n = \sum_{i,j,k} f_{ijk}\,\Delta^3 v_{ijk},\quad
\mathbf{u} = \frac{1}{n}\sum f_{ijk}\,\mathbf{v}_{ijk}\,\Delta^3 v_{ijk},\quad
P_{ab} = m\sum (v_a - u_a)(v_b - u_b)\,f_{ijk}\,\Delta^3 v_{ijk}.
$$

Each cell volume is $\Delta^3 v = v^2 \Delta v\,\Delta\Omega$, with $\Delta v / v = \tfrac{1}{2}\Delta E / E$ for a logarithmic energy grid. We include: / 각 셀 부피는 $\Delta^3 v = v^2 \Delta v \Delta\Omega$이며 로그 격자에서 $\Delta v / v = (1/2) \Delta E/E$. 다음을 포함한다.

- Per-anode relative efficiency $\varepsilon_{\rm rel}$. / 양극 상대 효율.
- Energy-dependent efficiency $\varepsilon_E(E)$ (leakage-field ramp at low $E$). / 저에너지 누설장 효과.
- Spacecraft-potential shift $E_\infty = E - q\Phi_{\rm sc}$. / 위성 전위에 의한 에너지 이동.

In [ ]:
def vdf_moments_3d(f_3d, energies_eV, polar_centers, az_centers, mass,
                   eps_rel=None, eps_E=None, phi_sc=0.0, q_sign=+1):
    """Compute (n, u, P) from a 3-D VDF on (E, polar, azimuth) bins.

    Args:
        f_3d: Phase-space density, shape (nE, nPolar, nAz), s^3/cm^6.
        energies_eV: Bin centers in eV (descending in our convention).
        polar_centers: Polar angles (rad), 0..pi.
        az_centers: Azimuth angles (rad), 0..2*pi.
        mass: Particle mass (kg).
        eps_rel: Per-anode relative efficiency, shape (nPolar,).
        eps_E: Energy-dependent efficiency, shape (nE,).
        phi_sc: Spacecraft potential (V); positive Phi reduces electron energy.
        q_sign: +1 for ions, -1 for electrons.

    Returns:
        Dict with 'n' (cm^-3), 'u' (km/s, 3-vector),
        'T_eV' (scalar temperature), 'P' (3x3 nPa).
    """
    if eps_rel is None:
        eps_rel = np.ones_like(polar_centers)
    if eps_E is None:
        eps_E = np.ones_like(energies_eV)

    # Apply spacecraft potential to bin energies.
    E_inf = energies_eV - q_sign * phi_sc
    valid = E_inf > 0
    E_use = np.where(valid, E_inf, 1e-30)

    # Apply efficiency division to f.
    eff_3d = eps_E[:, None, None] * eps_rel[None, :, None]
    f_corr = np.where(eff_3d > 0, f_3d / eff_3d, 0.0)
    f_corr = np.where(valid[:, None, None], f_corr, 0.0)

    v_cgs = np.sqrt(2.0 * E_use * EV_TO_J / mass) * 1e2  # cm/s
    dlogE = np.abs(np.gradient(np.log(np.where(E_use > 0, E_use, 1e-3))))
    dv = 0.5 * v_cgs * dlogE

    dpolar = np.gradient(polar_centers)
    daz = np.gradient(az_centers)
    sin_p = np.sin(polar_centers)
    domega = sin_p[:, None] * dpolar[:, None] * daz[None, :]

    w = (v_cgs ** 2 * dv)[:, None, None] * domega[None, :, :]

    vx_dir = np.sin(polar_centers)[:, None] * np.cos(az_centers)[None, :]
    vy_dir = np.sin(polar_centers)[:, None] * np.sin(az_centers)[None, :]
    vz_dir = np.cos(polar_centers)[:, None] * np.ones_like(az_centers)[None, :]

    n = np.sum(f_corr * w)
    if n <= 0:
        return {'n': 0.0, 'u': np.zeros(3), 'T_eV': 0.0, 'P': np.zeros((3, 3))}

    vx = v_cgs[:, None, None] * vx_dir[None, :, :]
    vy = v_cgs[:, None, None] * vy_dir[None, :, :]
    vz = v_cgs[:, None, None] * vz_dir[None, :, :]

    ux = np.sum(f_corr * w * vx) / n
    uy = np.sum(f_corr * w * vy) / n
    uz = np.sum(f_corr * w * vz) / n
    u = np.array([ux, uy, uz])  # cm/s

    dvx = vx - ux; dvy = vy - uy; dvz = vz - uz
    components = {
        (0, 0): dvx * dvx, (0, 1): dvx * dvy, (0, 2): dvx * dvz,
        (1, 1): dvy * dvy, (1, 2): dvy * dvz,
        (2, 2): dvz * dvz,
    }
    P = np.zeros((3, 3))
    for (a, b), arr in components.items():
        val = (mass * 1e3) * np.sum(arr * f_corr * w)
        P[a, b] = val * 0.1 * 1e9  # nPa
        if a != b:
            P[b, a] = P[a, b]

    T_eV = (P[0, 0] + P[1, 1] + P[2, 2]) / 3.0 / (n * 1e6 * Q_E) * 1e-9
    u_km_s = u * 1e-5
    return {'n': n, 'u': u_km_s, 'T_eV': T_eV, 'P': P}


def build_isotropic_vdf(grid, n, T_eV, mass, drift_kms=(0., 0., 0.)):
    """Build a drifting isotropic Maxwellian on a (nE, nPolar, nAz) grid."""
    nE = len(grid['energies']); nP = len(grid['polar_centers']); nA = len(grid['az_centers'])
    f = np.zeros((nE, nP, nA))
    drift_cgs = np.array(drift_kms) * 1e5
    T_J = T_eV * EV_TO_J
    norm = n * (mass / (2 * np.pi * T_J)) ** 1.5 * 1e-6
    for i, E in enumerate(grid['energies']):
        v_mag = np.sqrt(2.0 * E * EV_TO_J / mass) * 1e2
        for j, p in enumerate(grid['polar_centers']):
            for k, a in enumerate(grid['az_centers']):
                vx = v_mag * np.sin(p) * np.cos(a)
                vy = v_mag * np.sin(p) * np.sin(a)
                vz = v_mag * np.cos(p)
                v_rel2 = (vx - drift_cgs[0]) ** 2 + (vy - drift_cgs[1]) ** 2 + (vz - drift_cgs[2]) ** 2
                f[i, j, k] = norm * np.exp(-v_rel2 / (2 * T_J / mass * 1e4))
    return f


# Build a flowing magnetosheath ion VDF.
f_sheath = build_isotropic_vdf(grid_i, n=18.0, T_eV=180.0, mass=M_PROTON,
                               drift_kms=(-200.0, 30.0, 0.0))
mom = vdf_moments_3d(f_sheath, grid_i['energies'], grid_i['polar_centers'],
                     grid_i['az_centers'], M_PROTON)
print('Synthetic magnetosheath ion moments / 합성 자기초 이온 모멘트:')
print(f"  n  = {mom['n']:.2f} cm^-3 (input 18.0)")
print(f"  u  = ({mom['u'][0]:.1f}, {mom['u'][1]:.1f}, {mom['u'][2]:.1f}) km/s (input -200, 30, 0)")
print(f"  T  = {mom['T_eV']:.1f} eV (input 180)")
print(f"  Trace(P)/3 = {(mom['P'][0,0]+mom['P'][1,1]+mom['P'][2,2])/3:.3f} nPa")

### Spacecraft potential and energy-dependent efficiency / 위성 전위 + 에너지 의존 효율

McFadden §2.2 reports a $\sim$45% low-energy efficiency enhancement from MCP-front leakage with $e$-folding scale $E_0 \approx 180$ eV: $\varepsilon_E^{\rm i}(E) = 1 + \alpha\, e^{-E/E_0}$. Combining this with a positive $\Phi_{\rm sc}$ that pushes the lowest accessible electron energy upward demonstrates how moments shift when corrections are applied. / McFadden §2.2의 누설장 효과($\sim$45%, $E_0 \approx 180$ eV)와 양의 $\Phi_{\rm sc}$가 모멘트에 미치는 영향을 함께 보인다.

In [ ]:
# Energy-dependent ion efficiency model (McFadden eq. 4.9).
alpha = 0.45
E0 = 180.0
eps_E_ion = 1.0 + alpha * np.exp(-grid_i['energies'] / E0)

# Build a slightly cold electron VDF to see Phi_sc effect.
f_e_3d = build_isotropic_vdf(grid_e, n=12.0, T_eV=12.0, mass=M_ELECTRON)

mom_no_phi = vdf_moments_3d(f_e_3d, grid_e['energies'], grid_e['polar_centers'],
                            grid_e['az_centers'], M_ELECTRON, q_sign=-1, phi_sc=0.0)
mom_phi_5 = vdf_moments_3d(f_e_3d, grid_e['energies'], grid_e['polar_centers'],
                           grid_e['az_centers'], M_ELECTRON, q_sign=-1, phi_sc=5.0)
mom_phi_15 = vdf_moments_3d(f_e_3d, grid_e['energies'], grid_e['polar_centers'],
                            grid_e['az_centers'], M_ELECTRON, q_sign=-1, phi_sc=15.0)

print('Electron moments vs spacecraft potential / 위성 전위에 따른 전자 모멘트:')
print(f"  Phi_sc = 0 V  -> n = {mom_no_phi['n']:.2f} cm^-3, T = {mom_no_phi['T_eV']:.2f} eV")
print(f"  Phi_sc = 5 V  -> n = {mom_phi_5['n']:.2f} cm^-3, T = {mom_phi_5['T_eV']:.2f} eV")
print(f"  Phi_sc = 15 V -> n = {mom_phi_15['n']:.2f} cm^-3, T = {mom_phi_15['T_eV']:.2f} eV")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogx(grid_i['energies'], eps_E_ion, 'C0o-')
ax.axhline(1.0, color='k', linestyle=':', alpha=0.5)
ax.set_xlabel('Energy [eV]'); ax.set_ylabel(r'$\varepsilon_E^{\rm i}(E)$ enhancement factor')
ax.set_title('Ion sensor leakage-field efficiency / 이온 센서 누설장 효율 (McFadden Fig. 11c)')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

## Part 4: Plasma Sheet vs Lobe Population Separation / 자기꼬리 plasma sheet vs lobe 분리

THEMIS's primary science goal is substorm physics in the magnetotail. There the spacecraft routinely transit between the **plasma sheet** (warm, dense: $T_i \sim$ 1–10 keV, $n \sim$ 0.1–1 cm$^{-3}$) and the **lobe** (cold, tenuous: $T \lesssim 50$ eV, $n \lesssim 0.05$ cm$^{-3}$). After ESA cross-calibration the moments are reliable enough to classify each spin's measurement using a temperature/density threshold. / THEMIS의 주된 과학 목표는 자기꼬리 서브스톰. 자기꼬리에서 위성은 plasma sheet(따뜻하고 밀도 큼)와 lobe(차고 희박) 사이를 오간다. 교정된 모멘트가 정확하면 회전마다 온도·밀도로 분류 가능하다.

We synthesize a 30-minute boundary crossing as a smooth sigmoid in $(n, T)$, treat each spin as one measurement, and apply a simple two-feature classifier. / 30분짜리 합성 경계 통과를 시그모이드 $(n, T)$로 만들고 회전마다 모멘트를 두 특징 기반 분류기에 입력해 분리한다.

In [ ]:
def classify_population(n_cm3, T_eV):
    """Classify a plasma measurement as 'plasma_sheet', 'lobe', or 'boundary'.

    Rule of thumb from THEMIS magnetotail surveys:
        plasma sheet: T > 1 keV  AND  n > 0.1 cm^-3
        lobe:         T < 100 eV AND  n < 0.05 cm^-3
    Anything else falls into a 'boundary' transition class.
    """
    if T_eV > 1000.0 and n_cm3 > 0.1:
        return 'plasma_sheet'
    if T_eV < 100.0 and n_cm3 < 0.05:
        return 'lobe'
    return 'boundary'


# Generate a 30-minute synthetic crossing at 3 s spin cadence.
N_SPINS = 600
t = np.linspace(0.0, 30.0, N_SPINS)  # minutes
xi = 1.0 / (1.0 + np.exp(-(t - 15.0) / 1.2))  # 0 (lobe) -> 1 (sheet)

n_series = 0.02 + (0.6 - 0.02) * xi + 0.005 * np.random.randn(N_SPINS)
T_series = 40.0 + (4500.0 - 40.0) * xi + 50.0 * np.random.randn(N_SPINS)
n_series = np.maximum(n_series, 0.001)
T_series = np.maximum(T_series, 5.0)

labels = np.array([classify_population(n, T) for n, T in zip(n_series, T_series)])

color_map = {'plasma_sheet': 'C3', 'lobe': 'C0', 'boundary': 'C7'}
colors = [color_map[l] for l in labels]

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].semilogy(t, n_series, 'k-', alpha=0.5)
axes[0].scatter(t, n_series, c=colors, s=10)
axes[0].set_ylabel(r'$n$ [cm$^{-3}$]'); axes[0].grid(alpha=0.3)
axes[0].set_title('Synthetic plasma-sheet / lobe crossing (30 min) / 30분 plasma sheet–lobe 통과')

axes[1].semilogy(t, T_series, 'k-', alpha=0.5)
axes[1].scatter(t, T_series, c=colors, s=10)
axes[1].set_ylabel('T [eV]'); axes[1].grid(alpha=0.3)

class_counts = {k: int((labels == k).sum()) for k in ['lobe', 'boundary', 'plasma_sheet']}
class_pct = {k: 100.0 * v / N_SPINS for k, v in class_counts.items()}
axes[2].bar(list(class_pct.keys()), list(class_pct.values()),
            color=[color_map[k] for k in class_pct])
axes[2].set_ylabel('% of spins'); axes[2].set_xlabel('time [min] (bottom panel: class fraction)')
axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout(); plt.show()
print('Classification breakdown / 분류 결과:')
for k, v in class_counts.items():
    print(f"  {k:<12} {v:4d} spins  ({class_pct[k]:5.1f} %)")

In [ ]:
# Scatter view in (n, T) feature space - the classification regions.
fig, ax = plt.subplots(figsize=(8, 6))
for cls, col in color_map.items():
    mask = labels == cls
    ax.scatter(n_series[mask], T_series[mask], c=col, label=cls, s=14, alpha=0.7)

# Decision regions (visualize thresholds).
ax.axhline(1000.0, color='k', linestyle='--', alpha=0.4)
ax.axhline(100.0, color='k', linestyle='--', alpha=0.4)
ax.axvline(0.1, color='k', linestyle=':', alpha=0.4)
ax.axvline(0.05, color='k', linestyle=':', alpha=0.4)

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'$n$ [cm$^{-3}$]'); ax.set_ylabel('T [eV]')
ax.set_title('Plasma-sheet / lobe classifier in feature space / 특징 공간에서의 분류기')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

## Summary / 요약

| Concept / 개념 | This Notebook / 이 노트북 | THEMIS / Modern Equivalent / THEMIS · 현대 동등물 |
|---|---|---|
| Top-hat ESA bin geometry / 탑햇 ESA 빈 기하 | 32 log E × 16 polar × 16 azimuth grid; per-pixel $G$ / 32 로그 E × 16 polar × 16 방위 격자 | THEMIS iESA: 32 E × 16 anodes × 32 spin sectors, $G_{\rm in\text{-}flight}=0.0061\,{\rm cm^2\,sr\,eV}$ / THEMIS iESA 실측치 |
| Cross-calibration / 교차 교정 | Match $N_i / N_e \to 1$ in synthetic sheath; recover $\varepsilon_{\rm rel} = 0.88$ / 자기초에서 $N_i/N_e=1$로 맞춰 $\varepsilon_{\rm rel}=0.88$ 복원 | McFadden §2.5 anchored on THC iESA, propagated by sheath agreement / McFadden §2.5 — THC iESA 기준 전파 |
| Dead-time correction / 데드타임 보정 | $C_t = C_m / (1 - C_m \tau)$ at 170 ns / 170 ns 비포화 모형 | Amptek A121 pre-amp — same form / Amptek A121 동일 형태 |
| VDF moments with weighting / 가중치 모멘트 | Discrete sums $n, \mathbf{u}, P$ with $\varepsilon_{\rm rel}, \varepsilon_E, \Phi_{\rm sc}$ / $\varepsilon_{\rm rel}, \varepsilon_E, \Phi_{\rm sc}$ 적용 | Onboard moment packets compute the same on the IDPU / IDPU 온보드 모멘트 패킷에서 동일 계산 |
| Plasma-sheet / lobe split / plasma sheet 분리 | $T$ + $n$ thresholds on synthetic crossing / 합성 통과에 $T,n$ 임계값 | THEMIS substorm classifications follow same heuristics / THEMIS 서브스톰 분류 동일 방식 |

### Key takeaways from the implementation / 구현의 시사점

- The **largest numerical sensitivity** is to the per-energy efficiency model: a 12% bias in $\varepsilon_{\rm rel}$ propagates linearly into $N_i$, exactly the regime McFadden et al. found for pre-flight $G$. / 가장 큰 수치 민감도는 에너지별 효율 모형 — $\varepsilon_{\rm rel}$의 12% 편향이 $N_i$에 그대로 전이된다.
- Dead-time only matters when the per-pixel rate exceeds $\sim$100 kHz; for cold lobe plasmas it is negligible, but the 170 ns Amptek constant is essential for shock and sheath crossings. / 데드타임은 픽셀당 100 kHz 이상에서만 유의 — lobe에서는 무시 가능하나 충격파·자기초에서는 필수.
- Spacecraft potential of even 5–15 V meaningfully shifts cold electron moments because eESA's lowest bin is $\sim$6 eV; this is why THEMIS performs $\Phi_{\rm sc}$ correction *onboard* before computing moments. / 5–15 V의 위성 전위만으로도 차가운 전자 모멘트가 크게 바뀌므로 THEMIS는 온보드에서 보정한다.
- Classification of magnetotail populations needs only two features ($n$, $T$) once the moments are properly calibrated — confirming that ESA calibration quality directly controls substorm science. / 자기꼬리 분류는 모멘트가 잘 교정되면 두 특징만으로 충분 — ESA 교정 품질이 서브스톰 과학을 직접 지배한다는 의미.